# 📊 Painel de Análise de Vendas com Visualização de Dados
Projeto prático com **pandas**, **numpy**, **matplotlib** e **seaborn**.

Etapas:
1. Preparação e importação de dados
2. Análise exploratória e estatísticas
3. Visualizações com matplotlib e seaborn
4. Estilização e apresentação

## Etapa 1 — Preparação e Importação de Dados

**1)** Importação das bibliotecas.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Diretório onde todos os gráficos serão salvos (item 20)
os.makedirs('graficos', exist_ok=True)

def salvar_grafico(fig, nome):
    """Salva a figura em graficos/<nome>.png"""
    fig.savefig(os.path.join('graficos', nome), dpi=150, bbox_inches='tight')

**2)** Carregamento do arquivo `vendas.csv` (colunas: `data`, `loja`, `produto`, `quantidade`, `preco_unitario`).

Como os itens 13 e 15 pedem análises por **região** e por **categoria de produto**, o dataset também possui as colunas `regiao` e `categoria`.

> Se o arquivo `vendas.csv` não estiver no ambiente (por exemplo, no Colab), o código abaixo **cria o dataset fictício automaticamente**.

In [ ]:
def gerar_dataset(n=800, seed=42):
    rng = np.random.default_rng(seed)
    lojas = {'Loja Centro': 'Sudeste', 'Loja Norte Shopping': 'Norte',
             'Loja Sul Plaza': 'Sul', 'Loja Litoral': 'Nordeste',
             'Loja Cerrado': 'Centro-Oeste'}
    produtos = {  # produto: (categoria, preço base)
        'Notebook': ('Eletrônicos', 3200), 'Smartphone': ('Eletrônicos', 2100),
        'Fone de Ouvido': ('Acessórios', 180), 'Mouse': ('Acessórios', 90),
        'Teclado': ('Acessórios', 150), 'Monitor': ('Eletrônicos', 950),
        'Cadeira Gamer': ('Móveis', 1100), 'Mesa de Escritório': ('Móveis', 650),
        'Webcam': ('Acessórios', 220), 'Impressora': ('Eletrônicos', 780),
    }
    datas = pd.to_datetime('2024-01-01') + pd.to_timedelta(rng.integers(0, 366, n), unit='D')
    loja = rng.choice(list(lojas), n, p=[.28, .18, .22, .17, .15])
    produto = rng.choice(list(produtos), n)
    df = pd.DataFrame({
        'data': datas.strftime('%Y-%m-%d'),
        'loja': loja,
        'regiao': [lojas[l] for l in loja],
        'categoria': [produtos[p][0] for p in produto],
        'produto': produto,
        'quantidade': rng.integers(1, 11, n).astype(float),
        'preco_unitario': [round(produtos[p][1] * rng.uniform(0.9, 1.15), 2) for p in produto],
    })
    # Alguns outliers de quantidade e valores ausentes propositais
    idx = rng.choice(n, 6, replace=False)
    df.loc[idx, 'quantidade'] = rng.integers(25, 40, 6)
    df.loc[rng.choice(n, 20, replace=False), 'quantidade'] = np.nan
    df.loc[rng.choice(n, 20, replace=False), 'preco_unitario'] = np.nan
    return df.sort_values('data').reset_index(drop=True)

if not os.path.exists('vendas.csv'):
    gerar_dataset().to_csv('vendas.csv', index=False)

df = pd.read_csv('vendas.csv')

# Converte a coluna 'data' para datetime
df['data'] = pd.to_datetime(df['data'])
df.head()

**3)** Nova coluna `faturamento = quantidade * preco_unitario`.

In [ ]:
df['faturamento'] = df['quantidade'] * df['preco_unitario']
df.head()

**4)** Verificação da integridade do dataset.

In [ ]:
# Tipos de dados
df.info()

In [ ]:
# Valores ausentes por coluna
df.isnull().sum()

In [ ]:
# Resumo estatístico
df.describe()

**5)** Tratamento de nulos: preços e quantidades faltantes são substituídos pela **média da respectiva coluna**. O faturamento é recalculado em seguida.

In [ ]:
df['quantidade'] = df['quantidade'].fillna(df['quantidade'].mean())
df['preco_unitario'] = df['preco_unitario'].fillna(df['preco_unitario'].mean())

# Recalcula o faturamento com os dados tratados
df['faturamento'] = df['quantidade'] * df['preco_unitario']

print('Nulos restantes:')
print(df.isnull().sum())

## Etapa 2 — Análise Exploratória e Estatísticas

**6)** Faturamento total por loja (do maior para o menor).

In [ ]:
fat_loja = (df.groupby('loja')['faturamento'].sum()
              .sort_values(ascending=False))
fat_loja

**7)** Ticket médio por loja (faturamento ÷ número de vendas).

In [ ]:
ticket_medio = (df.groupby('loja')['faturamento'].sum() /
                df.groupby('loja')['faturamento'].count()).sort_values(ascending=False)
ticket_medio.round(2)

**8)** Os cinco produtos mais vendidos em quantidade.

In [ ]:
top5_produtos = (df.groupby('produto')['quantidade'].sum()
                   .sort_values(ascending=False).head(5))
top5_produtos

**10)** Coluna `mes` extraída da data (feita antes do item 9 para facilitar os agrupamentos).

In [ ]:
df['mes'] = df['data'].dt.month
df[['data', 'mes']].head()

**9)** Mês de maior e de menor faturamento, no formato **Mês/Ano**.

In [ ]:
meses_pt = {1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril', 5: 'Maio', 6: 'Junho',
            7: 'Julho', 8: 'Agosto', 9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'}

# Resumo mensal (usado também no gráfico de linhas)
resumo_mensal = (df.groupby(df['data'].dt.to_period('M'))['faturamento'].sum()
                   .reset_index())
resumo_mensal['mes_ano'] = resumo_mensal['data'].apply(lambda p: f"{meses_pt[p.month]}/{p.year}")

linha_max = resumo_mensal.loc[resumo_mensal['faturamento'].idxmax()]
linha_min = resumo_mensal.loc[resumo_mensal['faturamento'].idxmin()]

print(f"Maior faturamento: {linha_max['mes_ano']} (R$ {linha_max['faturamento']:,.2f})")
print(f"Menor faturamento: {linha_min['mes_ano']} (R$ {linha_min['faturamento']:,.2f})")

## Etapa 4 (parte) — Tema global
**17)** Definido aqui, antes dos gráficos, para ser aplicado a todos eles.

In [ ]:
sns.set_style('whitegrid')
sns.set_palette(sns.color_palette('deep'))
plt.rcParams['figure.dpi'] = 100

## Etapa 3 — Visualizações com matplotlib e seaborn

**11)** Gráfico de barras — Faturamento por loja.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cores = sns.color_palette('deep', len(fat_loja))
barras = ax.bar(fat_loja.index, fat_loja.values, color=cores, edgecolor='black', label='Faturamento total')
ax.bar_label(barras, labels=[f'R$ {v/1e6:.2f} mi' for v in fat_loja.values], padding=3)
ax.set_title('Faturamento Total por Loja (2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Loja')
ax.set_ylabel('Faturamento (R$)')
ax.legend()
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
salvar_grafico(fig, 'faturamento_por_loja.png')
plt.show()

**12)** Gráfico de linhas — Evolução mensal do faturamento (marcadores `'o'` e rótulos em cada ponto).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(resumo_mensal['mes_ano'], resumo_mensal['faturamento'], marker='o', linewidth=2, label='Faturamento mensal')
for x, y in zip(resumo_mensal['mes_ano'], resumo_mensal['faturamento']):
    ax.annotate(f'{y/1e3:,.0f} mil', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax.set_title('Evolução Mensal do Faturamento (2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Mês/Ano')
ax.set_ylabel('Faturamento (R$)')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
salvar_grafico(fig, 'evolucao_mensal_faturamento.png')
plt.show()

**13)** Gráfico de pizza — Distribuição do faturamento por região.

In [ ]:
fat_regiao = df.groupby('regiao')['faturamento'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(fat_regiao.values, labels=fat_regiao.index, autopct='%1.1f%%', startangle=90,
       colors=sns.color_palette('deep', len(fat_regiao)), wedgeprops={'edgecolor': 'white'})
ax.set_title('Distribuição do Faturamento por Região', fontsize=14, fontweight='bold')
plt.tight_layout()
salvar_grafico(fig, 'faturamento_por_regiao.png')
plt.show()

**14)** Gráfico de dispersão — Quantidade × Faturamento (seaborn), destacando padrões e *outliers*.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='quantidade', y='faturamento', hue='categoria', alpha=0.7, ax=ax)

# Destaca os outliers de quantidade (acima do limite de Tukey)
q1, q3 = df['quantidade'].quantile([.25, .75])
limite = q3 + 1.5 * (q3 - q1)
outliers = df[df['quantidade'] > limite]
ax.scatter(outliers['quantidade'], outliers['faturamento'], s=160, facecolors='none',
           edgecolors='red', linewidths=1.5, label=f'Outliers (qtd > {limite:.0f})')

ax.set_title('Relação entre Quantidade Vendida e Faturamento', fontsize=14, fontweight='bold')
ax.set_xlabel('Quantidade de itens vendidos')
ax.set_ylabel('Faturamento (R$)')
ax.legend(title='Categoria')
plt.tight_layout()
salvar_grafico(fig, 'dispersao_quantidade_faturamento.png')
plt.show()

**15)** Boxplot — Distribuição de preços unitários por categoria.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='categoria', y='preco_unitario', hue='categoria', palette='Set2', legend=True, ax=ax)
ax.set_title('Distribuição do Preço Unitário por Categoria de Produto', fontsize=14, fontweight='bold')
ax.set_xlabel('Categoria')
ax.set_ylabel('Preço unitário (R$)')
ax.legend(title='Categoria')
plt.tight_layout()
salvar_grafico(fig, 'boxplot_precos_categoria.png')
plt.show()

**16)** Heatmap — Correlação entre variáveis numéricas.

In [ ]:
corr = df[['quantidade', 'preco_unitario', 'faturamento', 'mes']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=.5, ax=ax)
ax.set_title('Matriz de Correlação entre Variáveis Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
salvar_grafico(fig, 'heatmap_correlacao.png')
plt.show()

## Etapa 4 — Estilização e Apresentação

**17)** Tema global (`whitegrid` + paleta `deep`) definido antes da Etapa 3 e aplicado a todos os gráficos.

**18)** Todos os gráficos acima possuem título descritivo, rótulos de eixos e legendas legíveis (sem textos genéricos como "Gráfico 1").

**19)** Layout com múltiplos gráficos lado a lado (`plt.subplots`): faturamento por loja e por região.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Esquerda: faturamento por loja
axes[0].bar(fat_loja.index, fat_loja.values, color=sns.color_palette('deep', len(fat_loja)), edgecolor='black')
axes[0].set_title('Faturamento por Loja', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Loja')
axes[0].set_ylabel('Faturamento (R$)')
axes[0].tick_params(axis='x', rotation=25)

# Direita: faturamento por região
axes[1].pie(fat_regiao.values, labels=fat_regiao.index, autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('deep', len(fat_regiao)), wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Faturamento por Região', fontsize=13, fontweight='bold')

fig.suptitle('Comparativo de Faturamento: Loja × Região', fontsize=16, fontweight='bold')
plt.tight_layout()
salvar_grafico(fig, 'layout_loja_regiao.png')
plt.show()

**20)** Exportação: todos os gráficos já foram salvos em `graficos/`. Abaixo é gerado o arquivo consolidado `resumo_analitico.csv`.

In [ ]:
partes = [
    pd.DataFrame({'secao': 'Faturamento por loja (R$)', 'item': fat_loja.index, 'valor': fat_loja.values.round(2)}),
    pd.DataFrame({'secao': 'Ticket médio por loja (R$)', 'item': ticket_medio.index, 'valor': ticket_medio.values.round(2)}),
    pd.DataFrame({'secao': 'Faturamento por região (R$)', 'item': fat_regiao.index, 'valor': fat_regiao.values.round(2)}),
    pd.DataFrame({'secao': 'Top 5 produtos (quantidade)', 'item': top5_produtos.index, 'valor': top5_produtos.values.round(0)}),
    pd.DataFrame({'secao': 'Faturamento mensal (R$)', 'item': resumo_mensal['mes_ano'], 'valor': resumo_mensal['faturamento'].round(2)}),
    pd.DataFrame({'secao': 'Destaques do ano (R$)',
                  'item': [f"Maior faturamento: {linha_max['mes_ano']}", f"Menor faturamento: {linha_min['mes_ano']}"],
                  'valor': [round(linha_max['faturamento'], 2), round(linha_min['faturamento'], 2)]}),
]
resumo_analitico = pd.concat(partes, ignore_index=True)
resumo_analitico.to_csv('resumo_analitico.csv', index=False, encoding='utf-8-sig')
resumo_analitico

### 📝 Observações a partir das visualizações

In [ ]:
loja_top, loja_baixo = fat_loja.index[0], fat_loja.index[-1]
reg_top = fat_regiao.index[0]
corr_qf = corr.loc['quantidade', 'faturamento']
corr_pf = corr.loc['preco_unitario', 'faturamento']

print(f"• Gráfico de barras: '{loja_top}' lidera o faturamento (R$ {fat_loja.iloc[0]:,.2f}), "
      f"enquanto '{loja_baixo}' tem o menor (R$ {fat_loja.iloc[-1]:,.2f}).")
print(f"• Gráfico de pizza: a região {reg_top} concentra {fat_regiao.iloc[0]/fat_regiao.sum():.1%} do faturamento total.")
print(f"• Gráfico de linhas: melhor mês = {linha_max['mes_ano']}; pior mês = {linha_min['mes_ano']}.")
print(f"• Dispersão: {len(outliers)} vendas com quantidade atípica (> {limite:.0f} itens) aparecem como outliers.")
print(f"• Boxplot: Eletrônicos têm os maiores preços e maior dispersão; Acessórios, os menores e mais concentrados.")
print(f"• Heatmap: correlação do faturamento com quantidade = {corr_qf:.2f} e com preço unitário = {corr_pf:.2f}.")
print(f"• Top 5 produtos por quantidade: {', '.join(top5_produtos.index)}.")